# Resource Adequacy Metrics

`assetra` ships four resource adequacy metrics. They all summarise the same
underlying simulation output, but they measure different things, and a system
can look adequate under one and inadequate under another.

This notebook covers:

1. What the net hourly capacity matrix contains, and how shortfall appears in it.
2. The four built-in metrics and what each one counts.
3. A controlled example showing how the metrics come apart.
4. A larger probabilistic system, with plots of where and how risk accumulates.
5. How each metric responds as capacity is added.

For a worked end-to-end study using real system data, see the CISO example.
This notebook is deliberately small and self-contained so that the behaviour of
each metric can be seen directly.

## Setup

Plotting uses `matplotlib`, which is not a dependency of `assetra`.

In [ ]:
# !pip install matplotlib

import numpy as np
import matplotlib.pyplot as plt

from assetra.system import EnergySystemBuilder
from assetra.simulation import ProbabilisticSimulation
from assetra.units import DemandUnit, StaticUnit, StochasticUnit
from assetra.metrics import (
    ExpectedUnservedEnergy,
    LossOfLoadHours,
    LossOfLoadDays,
    LossOfLoadFrequency,
)
from assetra.utils import get_hourly_time_series_xr

plt.style.use("ggplot")

## 1. Where the metrics come from

Every metric reads the same object: the **net hourly capacity matrix** of a
`ProbabilisticSimulation`. This is a two-dimensional array of net system
capacity, with one row per Monte Carlo trial and one column per hour.

Net capacity is the sum of all unit contributions, with demand entering as a
negative contribution. **Negative values are shortfall hours**, where demand
exceeded available capacity. Every metric below is a different way of
summarising the negative entries in that matrix.

To make the behaviour visible, start with a system that has no randomness at
all: a constant 100 MW static unit against a demand profile we control exactly.
With no stochastic units, one trial is enough, and the shortfall pattern is
exactly what we wrote into the demand profile.

In [ ]:
DAYS = 10
HOURS = DAYS * 24
CAPACITY = 100.0  # MW
BASE_DEMAND = 90.0  # MW


def build_deterministic_system(hourly_demand):
    """Return a system with one static unit and the given demand profile."""
    builder = EnergySystemBuilder()
    builder.add_unit(DemandUnit(0, get_hourly_time_series_xr(hourly_demand)))
    builder.add_unit(
        StaticUnit(1, CAPACITY, get_hourly_time_series_xr([CAPACITY] * HOURS))
    )
    return builder.build()


def simulate(hourly_demand, trial_size=1):
    """Build, assign, and run a simulation for the given demand profile."""
    simulation = ProbabilisticSimulation(
        "2019-01-01 00:00:00", "2019-01-10 23:00:00", trial_size
    )
    simulation.assign_energy_system(build_deterministic_system(hourly_demand))
    simulation.run()
    return simulation


# six isolated shortfall hours, one on each of six different days
scattered_demand = np.full(HOURS, BASE_DEMAND)
for day in range(6):
    scattered_demand[day * 24 + 18] = 110.0

simulation = simulate(scattered_demand)
simulation

The `__repr__` confirms the simulation has been run. Now look at the net
capacity itself. Hours sitting at +10 MW are surplus; the six dips to -10 MW
are the shortfall hours.

In [ ]:
net_capacity = simulation.net_hourly_capacity_matrix

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(net_capacity.time, net_capacity[0], color="tab:blue", linewidth=1.2)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Net capacity (MW)")
ax.set_title("Net hourly capacity, six isolated shortfall hours")
ax.legend()
plt.tight_layout()
plt.show()

## 2. The four metrics

Each metric is instantiated with a simulation and evaluated with `evaluate()`.
All four are averages across Monte Carlo trials, reported per study horizon.

| Metric | Counts | Units |
| --- | --- | --- |
| `ExpectedUnservedEnergy` | Total energy not served | energy (MWh here) |
| `LossOfLoadHours` | Hours containing a shortfall | hours |
| `LossOfLoadDays` | Days containing at least one shortfall hour | days |
| `LossOfLoadFrequency` | Distinct shortfall events, each a contiguous run of hours | events |

The distinction that matters: EUE measures **magnitude**, LOLH measures
**duration**, LOLD and LOLF measure **how often trouble occurs** rather than how
long it lasts.

In [ ]:
def evaluate_all(simulation):
    """Return every built-in resource adequacy metric for a simulation."""
    return {
        "EUE (MWh)": ExpectedUnservedEnergy(simulation).evaluate(),
        "LOLH (h)": LossOfLoadHours(simulation).evaluate(),
        "LOLD (d)": LossOfLoadDays(simulation).evaluate(),
        "LOLF (events)": LossOfLoadFrequency(simulation).evaluate(),
    }


evaluate_all(simulation)

Six isolated one-hour shortfalls of 10 MW each, so: 60 MWh unserved, 6 shortfall
hours, spread across 6 separate days, forming 6 separate events. With this
pattern every count happens to coincide. The next section breaks that.

## 3. Why you need more than one metric

Two systems can be identical under one metric and very different under another.
Below are three demand profiles, each producing exactly **six** shortfall hours:

- **Scattered** — six isolated hours on six different days, 10 MW deep.
- **Clustered** — the same six hours arranged as two three-hour blocks on two days.
- **Scattered, deep** — the same pattern as scattered, but 40 MW deep instead of 10 MW.

In [ ]:
# same six shortfall hours, arranged as two three-hour blocks on two days
clustered_demand = np.full(HOURS, BASE_DEMAND)
for day in (1, 5):
    for hour in range(17, 20):
        clustered_demand[day * 24 + hour] = 110.0

# six isolated hours again, but four times deeper
deep_demand = np.full(HOURS, BASE_DEMAND)
for day in range(6):
    deep_demand[day * 24 + 18] = 130.0

profiles = {
    "scattered": scattered_demand,
    "clustered": clustered_demand,
    "scattered, deep": deep_demand,
}

results = {name: evaluate_all(simulate(p)) for name, p in profiles.items()}

header = f"{'profile':<18}" + "".join(f"{k:>16}" for k in results["scattered"])
print(header)
print("-" * len(header))
for name, values in results.items():
    print(f"{name:<18}" + "".join(f"{v:>16.1f}" for v in values.values()))

Read across the rows:

- **Scattered vs clustered**: identical EUE and identical LOLH, but LOLD and LOLF
  drop from 6 to 2. The same amount of unserved energy over the same number of
  hours, delivered as two long events instead of six short ones. Operationally
  these are very different situations, and only LOLD and LOLF distinguish them.
- **Scattered vs scattered, deep**: identical LOLH, LOLD, and LOLF, but EUE
  triples. The shortfalls happen exactly as often and for exactly as long; they
  are simply deeper. Only EUE sees this.

This is why a target expressed purely as "one day in ten years" says nothing
about how severe those events are, and why EUE is usually reported alongside it.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True, sharey=True)

for ax, (name, profile) in zip(axes, profiles.items()):
    net = simulate(profile).net_hourly_capacity_matrix[0]
    ax.plot(net.time, net, color="tab:blue", linewidth=1.0)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_ylabel("MW")
    ax.set_title(name, fontsize=10, loc="left")

plt.tight_layout()
plt.show()

## 4. A probabilistic system

The deterministic examples above make the definitions legible, but real studies
are probabilistic: capacity is available only some of the time, so shortfall
appears in some trials and not others.

The system below runs a full year against a seasonal, daily-peaking demand
profile, served by a fleet of stochastic units each with a 6% forced outage
rate. The fleet is deliberately sized a little tight, so that shortfall is
frequent enough to plot clearly.

In [ ]:
YEAR_HOURS = 8760
TRIAL_SIZE = 100
FLEET_SIZE = 146
UNIT_CAPACITY = 1.0  # MW
FORCED_OUTAGE_RATE = 0.06

hours = np.arange(YEAR_HOURS)
day_of_year = hours / 24.0

# seasonal swing, daily cycle, and a summer afternoon peak
annual_demand = (
    100
    + 18 * np.sin(2 * np.pi * (day_of_year - 172) / 365)
    - 8 * np.cos(2 * np.pi * hours / 24)
    + 12
    * np.clip(np.sin(2 * np.pi * (day_of_year - 172) / 365), 0, None)
    * np.cos(2 * np.pi * (hours - 15) / 24)
)

builder = EnergySystemBuilder()
builder.add_unit(DemandUnit(0, get_hourly_time_series_xr(annual_demand)))
for unit_id in range(1, FLEET_SIZE + 1):
    builder.add_unit(
        StochasticUnit(
            unit_id,
            UNIT_CAPACITY,
            get_hourly_time_series_xr([UNIT_CAPACITY] * YEAR_HOURS),
            get_hourly_time_series_xr([FORCED_OUTAGE_RATE] * YEAR_HOURS),
        )
    )

energy_system = builder.build()
energy_system

In [ ]:
annual_simulation = ProbabilisticSimulation(
    "2019-01-01 00:00:00", "2019-12-31 23:00:00", TRIAL_SIZE
)
annual_simulation.assign_energy_system(energy_system)
annual_simulation.run()

annual_metrics = evaluate_all(annual_simulation)
for name, value in annual_metrics.items():
    print(f"{name:<16}{value:>10.2f}")

Two ratios are worth computing from these numbers directly, because they
describe the *shape* of the risk rather than its size:

- **LOLH / LOLF** is the average event duration in hours.
- **LOLH / LOLD** is the average number of shortfall hours on a day that has any.

In [ ]:
print(f"average event duration: {annual_metrics['LOLH (h)'] / annual_metrics['LOLF (events)']:.2f} h")
print(f"shortfall hours per affected day: {annual_metrics['LOLH (h)'] / annual_metrics['LOLD (d)']:.2f} h")

### Where does risk occur?

A single number hides *when* the system is short. Because the net hourly
capacity matrix keeps the full time dimension, the share of trials with a
shortfall in each hour can be plotted directly. This hourly risk profile is
often more actionable than any scalar metric.

In [ ]:
annual_net = annual_simulation.net_hourly_capacity_matrix
shortfall = annual_net < 0

# probability of shortfall by hour of day, and by month
risk_by_hour = shortfall.groupby("time.hour").mean(dim=["trial", "time"])
risk_by_month = shortfall.groupby("time.month").mean(dim=["trial", "time"])

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.5))

left.bar(risk_by_hour.hour, risk_by_hour * 100, color="tab:red", alpha=0.8)
left.set_xlabel("Hour of day")
left.set_ylabel("Trials with shortfall (%)")
left.set_title("Risk by hour of day", fontsize=10, loc="left")

right.bar(risk_by_month.month, risk_by_month * 100, color="tab:red", alpha=0.8)
right.set_xlabel("Month")
right.set_ylabel("Trials with shortfall (%)")
right.set_title("Risk by month", fontsize=10, loc="left")

plt.tight_layout()
plt.show()

Risk concentrates in summer afternoons, which is where the demand profile peaks.
Note that this is a property of the *system*, not of any one metric: all four
metrics summarise this same pattern, and all four would fall if capacity were
added specifically for those hours.

### How much does the answer vary between trials?

Every metric reported above is an average across trials, and that average
conceals the spread it was drawn from. Plotting per-trial unserved energy shows
how much an individual year can differ from the reported expectation.

In [ ]:
unserved_by_trial = -annual_net.where(annual_net < 0, 0).sum(dim="time")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(unserved_by_trial, bins=25, color="tab:blue", alpha=0.8)
ax.axvline(
    float(unserved_by_trial.mean()),
    color="black",
    linestyle="--",
    label=f"EUE = {float(unserved_by_trial.mean()):.0f} MWh",
)
ax.set_xlabel("Unserved energy in a single trial (MWh)")
ax.set_ylabel("Number of trials")
ax.set_title("Distribution of unserved energy across trials", fontsize=10, loc="left")
ax.legend()
plt.tight_layout()
plt.show()

print(f"median trial: {float(unserved_by_trial.median()):.0f} MWh")
print(f"worst trial:  {float(unserved_by_trial.max()):.0f} MWh")
print(f"trials with no shortfall: {int((unserved_by_trial == 0).sum())} of {TRIAL_SIZE}")

In this system the spread is moderate and roughly symmetric, and no trial
escapes shortfall entirely. That is a consequence of how tightly the fleet is
sized: shortfall here is common, so each trial accumulates many small events and
the annual total concentrates around its mean.

A system sized to a realistic adequacy target behaves very differently. When
shortfall is rare, most trials contain none at all and the total is dominated by
a handful of severe trials, giving a sharply right-skewed distribution. In that
regime the mean is estimated from very few informative trials, and `trial_size`
must be much larger for the reported metric to be stable. Increasing
`trial_size` reduces uncertainty in the *mean*; it does not reduce the spread
itself, which is a real property of the system.

## 5. Choosing a metric

| If you care about | Use | Because |
| --- | --- | --- |
| Total energy at risk, or cost of shortfall | `ExpectedUnservedEnergy` | Only metric sensitive to shortfall depth |
| Comparing against an hours-based standard | `LossOfLoadHours` | Counts event-hours per horizon |
| Comparing against a "one day in N years" standard | `LossOfLoadDays` | Counts event-days, matching how targets are written |
| Operational burden, or storage adequacy | `LossOfLoadFrequency` | Counts distinct events, and with LOLH gives event duration |

In practice these are reported together. The controlled example in section 3
shows why: any one of them alone can be held constant while the system's actual
risk profile changes substantially.